# TC3-38 – Auditoria da Feature Table ml_aluno

**Responsável:** Sonia Andreia Ribeiro

**Sprint:** Fase 2

**Jira:** TC3-38, TC3-39

## Objetivo

Validar a qualidade da Feature Table `ml_aluno`, verificando sua integridade, estrutura e consistência antes da utilização nos modelos de Machine Learning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Configuração dos gráficos
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 2. Configuração do ambiente

Nesta etapa definimos o caminho da Feature Table e localizamos automaticamente todos os arquivos Parquet das partições de 2023 e 2024.

In [ ]:
import zipfile
from pathlib import Path

# Localiza a raiz do projeto a partir do diretório atual do notebook.
# Assim o notebook funciona no VS Code, Jupyter local ou outro clone do repositório.
def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dados" / "principal" / "ml_aluno.zip").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível encontrar dados/principal/ml_aluno.zip. "
        "Execute o notebook dentro do repositório ou confira se os dados foram baixados."
    )

PROJECT_ROOT = encontrar_raiz_projeto()
ZIP_PATH = PROJECT_ROOT / "dados" / "principal" / "ml_aluno.zip"
EXTRACT_PATH = PROJECT_ROOT / "dados" / "principal"

if not (EXTRACT_PATH / "ml_aluno").exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

print("Arquivo zip:", ZIP_PATH)
print("Pasta de extração:", EXTRACT_PATH)


In [ ]:
# Caminho local da feature table extraída
BASE_PATH = EXTRACT_PATH / "ml_aluno"

# Partições da Gold
PATH_2023 = BASE_PATH / "ano2023"
PATH_2024 = BASE_PATH / "ano2024"

print("2023:", PATH_2023)
print("2024:", PATH_2024)


In [ ]:
for item in BASE_PATH.iterdir():
    print(item.name)

## 3. Localização dos arquivos Parquet

A Feature Table está particionada por ano. Vamos identificar automaticamente todos os arquivos disponíveis.

In [ ]:
arquivos_2023 = sorted(PATH_2023.glob("*.parquet"))
arquivos_2024 = sorted(PATH_2024.glob("*.parquet"))

print(f"Arquivos 2023: {len(arquivos_2023)}")
print(f"Arquivos 2024: {len(arquivos_2024)}")

### 3.1 Conferência das partições

Validação da quantidade de arquivos Parquet por partição da Feature Table.

In [ ]:
print("===== 2023 =====")
for arq in arquivos_2023:
    print(arq.name)

print("\n===== 2024 =====")
for arq in arquivos_2024:
    print(arq.name)

In [ ]:
faltantes = set(a.name for a in arquivos_2023) - set(a.name for a in arquivos_2024)

print("Arquivos existentes em 2023 e ausentes em 2024:")
for arq in sorted(faltantes):
    print(arq)

### Achado 01 — Inconsistência entre as partições

Foi identificada divergência na quantidade de arquivos Parquet entre as partições anuais da Feature Table `ml_aluno`.

- **2023:** 36 arquivos
- **2024:** 33 arquivos

Foi realizada a comparação dos nomes dos arquivos para identificar os objetos ausentes.

In [ ]:
faltantes_df = pd.DataFrame(
    sorted(faltantes),
    columns=["Arquivo ausente em 2024"]
)

faltantes_df

In [ ]:
df_2023 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2023],
    ignore_index=True
)

df_2024 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2024],
    ignore_index=True
)

print(f"Registros 2023: {len(df_2023):,}")
print(f"Registros 2024: {len(df_2024):,}")

## 4. Consolidação da Feature Table

As partições anuais são unificadas para formar a Feature Table completa (`ml_aluno`), que será utilizada nas etapas de EDA, engenharia de features e modelagem.

In [ ]:
# Consolidação da base
df = pd.concat([df_2023, df_2024], ignore_index=True)

print("=" * 40)
print("FEATURE TABLE CONSOLIDADA")
print("=" * 40)
print(f"Registros: {len(df):,}")
print(f"Colunas: {df.shape[1]}")

## 5. Inventário da Feature Table

Resumo estrutural da base consolidada utilizada na modelagem de Machine Learning.

In [ ]:
resumo = pd.DataFrame({
    "Métrica": [
        "Registros",
        "Colunas",
        "Variáveis numéricas",
        "Variáveis categóricas"
    ],
    "Valor": [
        len(df),
        df.shape[1],
        len(df.select_dtypes(include="number").columns),
        len(df.select_dtypes(exclude="number").columns)
    ]
})

resumo


## 6. Estrutura das variáveis

Validação dos tipos de dados, memória utilizada e consistência estrutural da Feature Table.

In [ ]:
# Informações estruturais
df.info()

In [ ]:
dicionario = pd.DataFrame({
    "Variável": df.columns,
    "Tipo": df.dtypes.astype(str),
    "Nulos": df.isna().sum().values,
    "% Nulos": (df.isna().mean()*100).round(2).values,
    "Valores únicos": df.nunique().values
})

dicionario.sort_values("% Nulos", ascending=False)

## 6. Auditoria de valores nulos

Classificação das variáveis conforme o percentual de valores ausentes, identificando possíveis riscos para a modelagem.

In [ ]:
# Apenas variáveis com nulos
nulos = (
    dicionario[dicionario["Nulos"] > 0]
    .sort_values("% Nulos", ascending=False)
    .reset_index(drop=True)
)

nulos

In [ ]:
plt.figure(figsize=(8,4))

plt.barh(
    nulos["Variável"],
    nulos["% Nulos"]
)

plt.xlabel("% de valores nulos")
plt.title("Variáveis com valores ausentes")
plt.gca().invert_yaxis()

plt.show()

In [ ]:
duplicados = df["id_aluno"].duplicated().sum()

print(f"IDs de aluno duplicados: {duplicados:,}")
print(f"Alunos únicos: {df['id_aluno'].nunique():,}")

## 7. Validação da granularidade

Verificação da regra de negócio: uma linha representa um aluno ou uma observação do aluno?

In [ ]:
ocorrencias = (
    df.groupby("id_aluno")
      .size()
      .value_counts()
      .sort_index()
      .reset_index()
)

ocorrencias.columns = ["Quantidade de registros por aluno", "Número de alunos"]

ocorrencias

In [ ]:
plt.figure(figsize=(7,4))

plt.bar(
    ocorrencias["Quantidade de registros por aluno"].astype(str),
    ocorrencias["Número de alunos"]
)

plt.title("Distribuição de registros por aluno")
plt.xlabel("Registros por aluno")
plt.ylabel("Quantidade de alunos")

plt.show()

### Achado 04 — Granularidade da Feature Table

A validação da chave `id_aluno` demonstrou que a Feature Table possui natureza longitudinal.

**Conclusão:** a unidade analítica é uma observação anual do aluno, e não um registro único por aluno.

In [ ]:
plt.figure(figsize=(6,4))

ocorrencias.plot(
    kind="bar",
    x="Quantidade de registros por aluno",
    y="Número de alunos",
    legend=False,
    color="#2563EB"
)

plt.title("Granularidade da Feature Table")
plt.xlabel("Registros por aluno")
plt.ylabel("Quantidade de alunos")
plt.xticks(rotation=0)
plt.tight_layout()

plt.show()

In [ ]:
validacao = (
    df.groupby("id_aluno")["ano_referencia"]
      .nunique()
      .value_counts()
      .sort_index()
)

validacao

### Achado 04 — Validação da Granularidade

A análise da chave `id_aluno` demonstrou que a Feature Table possui natureza longitudinal.

Dos **2.352.328 alunos únicos**, **1.515.671** possuem registros em dois anos de referência (2023 e 2024), enquanto **836.657** aparecem em apenas um ano.

**Conclusão:** a unidade analítica da `ml_aluno` é **1 linha = 1 aluno por ano de referência**, e não um registro único por aluno.

## 8. Distribuição da variável alvo

Validação do balanceamento da variável `alfabetizado`, identificando a proporção entre as classes positivas e negativas.

In [ ]:
target = (
    df["alfabetizado"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Classe")
    .reset_index(name="Quantidade")
)

target["Percentual"] = (
    target["Quantidade"] / target["Quantidade"].sum() * 100
).round(2)

target

### Distribuição da variável `alfabetizado`

Análise do balanceamento entre as classes da variável alvo.

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    target["Classe"].astype(str),
    target["Quantidade"],
    color=["#64748B", "#0EA5A4"]
)

plt.title("Distribuição da variável alvo")
plt.xlabel("Classe (0 = Não alfabetizado | 1 = Alfabetizado)")
plt.ylabel("Quantidade de alunos")

for i, valor in enumerate(target["Quantidade"]):
    plt.text(i, valor + 20000, f"{valor:,}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
split = (
    df["dataset_split"]
    .value_counts()
    .rename_axis("Conjunto")
    .reset_index(name="Quantidade")
)

split["Percentual"] = (
    split["Quantidade"] / split["Quantidade"].sum() * 100
).round(2)

split

### Achado 06 — Validação do Dataset Split

A Feature Table apresenta uma estratégia de particionamento consistente para modelagem supervisionada.

| Conjunto | Percentual |
|----------|-----------:|
| Treino | 69,94% |
| Validação | 15,02% |
| Teste | 15,04% |

**Conclusão:** a divisão encontra-se balanceada e adequada para treinamento, ajuste de hiperparâmetros e avaliação final do modelo.

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    split["Conjunto"],
    split["Quantidade"],
    color=["#2563EB", "#0EA5A4", "#64748B"]
)

plt.title("Distribuição do Dataset Split")
plt.ylabel("Quantidade de registros")

for i, valor in enumerate(split["Quantidade"]):
    plt.text(i, valor + 30000, f"{split['Percentual'][i]}%", ha="center")

plt.tight_layout()
plt.show()

# 9. Parecer Técnico da Auditoria

## Status da Feature Table

🟢 **APTA PARA MODELAGEM**

### Principais evidências

- A Feature Table consolidada possui **3.867.999 registros** e **19 variáveis**.
- A granularidade foi validada como **1 linha = 1 aluno por ano de referência**.
- A variável alvo `alfabetizado` encontra-se balanceada (48,69% × 51,31%).
- O `dataset_split` apresenta distribuição adequada (70% / 15% / 15%).
- Foram identificadas quatro variáveis com valores ausentes, que deverão ser avaliadas na etapa de pré-processamento (TC3-49).

### Próximas etapas

- TC3-40 — Inventário e análise das variáveis.
- TC3-41 — Definição das features.
- TC3-49 — Estratégia de tratamento de nulos e engenharia de atributos.